# 03. Modeling

На данном этапе проводится исследование способов улучшения baseline-модели.

В качестве исходной модели используется логистическая регрессия,
полученная на предыдущем этапе.

Основные направления экспериментов:

1. настройка гиперпараметров Logistic Regression;
2. изменение параметров регуляризации;
3. настройка порога классификации;
4. сравнение с другими алгоритмами классификации.

Качество моделей сравнивается с результатами baseline.

## 0. Импорт библиотек

Импортируем библиотеки, необходимые для подготовки данных,
обучения моделей и оценки их качества.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 1. Загрузка и подготовка данных

На данном этапе загружаем исходный датасет и выполняем минимальную
подготовку данных, необходимую для построения моделей.

EDA в этом ноутбуке повторно не выполняется, поскольку исследование
данных было проведено в `01_eda.ipynb`.

In [3]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.shape

(7043, 21)

### 1.1 Подготовка TotalCharges

Признак `TotalCharges` содержит числовые значения, представленные в
строковом формате. Поэтому преобразуем его в числовой тип.

Параметр `errors="coerce"` заменяет значения, которые невозможно
преобразовать в число, на `NaN`.

In [4]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df.loc[
    df["TotalCharges"].isna() & (df["tenure"] == 0),
    "TotalCharges"
] = 0

In [5]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [6]:
df["TotalCharges"].dtype

dtype('float64')

### 1.2 Формирование признаков и целевой переменной

Целевой переменной является `Churn`.

Преобразуем её значения:

- `No` → 0;
- `Yes` → 1.

Идентификатор клиента `customerID` исключается из признаков,
поскольку он не содержит полезной информации для прогнозирования
оттока.

In [7]:
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

X = df.drop(
    columns=["Churn", "customerID"]
)

In [8]:
X.shape, y.shape

((7043, 19), (7043,))

### 1.3 Разделение данных

Разделим данные на обучающую и тестовую выборки.

80% данных используется для обучения модели, а 20% — для
финальной проверки на отложенной тестовой выборке.

Параметр `stratify=y` сохраняет примерно одинаковое соотношение
классов `Churn` в обучающей и тестовой выборках.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [10]:
X_train.shape, X_test.shape

((5634, 19), (1409, 19))

### 1.4 Разделение признаков по типу

Числовые и категориальные признаки требуют разной предварительной
обработки.

Числовые признаки будут стандартизированы с помощью `StandardScaler`.

Категориальные признаки будут преобразованы в бинарные признаки
с помощью `OneHotEncoder`.

In [11]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

### 1.5 Предварительная обработка признаков

Для числовых признаков используется стандартизация:

$$
z = \frac{x-\mu}{\sigma}
$$

Для категориальных признаков используется One-Hot Encoding.

Предварительная обработка объединяется в `ColumnTransformer`, чтобы
каждый тип признаков обрабатывался соответствующим способом.

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

## 2. Baseline

В качестве исходной модели используется логистическая регрессия
без дополнительной настройки гиперпараметров.

Результаты baseline:

| Метрика | Значение |
|---|---:|
| Accuracy | 0.807 |
| Precision | 0.660 |
| Recall | 0.561 |
| F1-score | 0.607 |
| ROC-AUC | 0.842 |

Эти значения используются как точка отсчёта для последующих экспериментов.

## 3. Настройка регуляризации

Параметр $C$ определяет силу регуляризации логистической регрессии.

Малое значение $C$ соответствует более сильной регуляризации,
а большое значение $C$ — более слабой.

Проверим несколько значений:

$$
C \in \{0.01,\ 0.1,\ 1,\ 10,\ 100\}
$$

Для каждого значения обучим отдельную модель и сравним её качество
с baseline.

In [13]:
C_values = [0.01, 0.1, 1, 10, 100]

result_C = []

for C in C_values:
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                C=C,
                max_iter=10000
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred_C = model.predict(X_test)
    y_proba_C = model.predict_proba(X_test)[:, 1]

    result_C.append({
        "C": C,
        "Accuracy": accuracy_score(y_test, y_pred_C),
        "Precision": precision_score(y_test, y_pred_C),
        "Recall": recall_score(y_test, y_pred_C),
        "F1": f1_score(y_test, y_pred_C),
        "ROC-AUC": roc_auc_score(y_test, y_proba_C)
    }
    )

result_C = pd.DataFrame(result_C)
result_C

,C,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.01,0.794890,0.656827,0.475936,0.551938,0.837952
1,0.10,0.800568,0.649518,0.540107,0.589781,0.841383
2,1.00,0.806955,0.660377,0.561497,0.606936,0.842171
3,10.00,0.804826,0.655172,0.558824,0.603175,0.841311
4,100.00,0.799858,0.644654,0.548128,0.592486,0.840776


### 3.1 Результаты эксперимента

Проверка различных значений параметра $C$ не привела к существенному
улучшению baseline-модели.

При малом значении $C = 0.01$, соответствующем сильной регуляризации,
наблюдается снижение Recall и F1-score:

$$
Recall = 0.476
$$

$$
F_1 = 0.552
$$

При $C = 1$ получены максимальные значения среди проверенных
конфигураций:

$$
F_1 = 0.607
$$

$$
ROC\text{-}AUC = 0.842
$$

Увеличение $C$ до 10 и 100 не привело к улучшению качества.

Таким образом, изменение силы L2-регуляризации в рассмотренном диапазоне
не позволило получить заметного улучшения относительно baseline.
Значение $C = 1$ сохраняется в качестве текущей конфигурации.

## 4. Учет дисбаланса классов

В целевой переменной наблюдается дисбаланс классов:

- `Churn = 0` — около 73.5%;
- `Churn = 1` — около 26.5%.

Поэтому проверим влияние параметра `class_weight`.

При стандартном значении:

$$
class\_weight = None
$$

ошибки обоих классов учитываются с одинаковым весом.

При:

$$
class\_weight = balanced
$$

веса классов автоматически рассчитываются обратно пропорционально
их частоте в обучающей выборке.

Таким образом, менее представленный класс получает больший вес.

In [14]:
class_weights = [None, 'balanced']

result_weights = []

for weight in class_weights:

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=1,
            class_weight=weight,
            max_iter=10000
        ))
    ])

    model.fit(X_train, y_train)

    y_pred_weight = model.predict(X_test)
    y_proba_weight = model.predict_proba(X_test)[:, 1]

    result_weights.append({
        "class_weight": weight,
        "Accuracy": accuracy_score(y_test, y_pred_weight),
        "Precision": precision_score(y_test, y_pred_weight),
        "Recall": recall_score(y_test, y_pred_weight),
        "F1": f1_score(y_test, y_pred_weight),
        "ROC-AUC": roc_auc_score(y_test, y_proba_weight)
    }
    )

result_weights = pd.DataFrame(result_weights)
result_weights

,class_weight,Accuracy,Precision,Recall,F1,ROC-AUC
0,NaN,0.806955,0.660377,0.561497,0.606936,0.842171
1,balanced,0.738822,0.505172,0.783422,0.614256,0.841750


### 4.1 Результаты эксперимента

Использование `class_weight="balanced"` существенно изменило поведение
модели.

Recall увеличился:

$$
0.561 \rightarrow 0.783
$$

Это означает, что модель стала обнаруживать значительно большую долю
реально ушедших клиентов.

Одновременно Precision снизился:

$$
0.660 \rightarrow 0.505
$$

Это связано с увеличением количества клиентов, которых модель ошибочно
относит к классу `Churn = 1`.

F1-score немного увеличился:

$$
0.607 \rightarrow 0.614
$$

При этом ROC-AUC практически не изменился:

$$
0.842 \rightarrow 0.842
$$

Таким образом, изменение весов классов в основном повлияло на баланс
между ошибками классов при выбранном пороге классификации, а не на
общую способность модели разделять два класса.

## 5. Настройка порога классификации

По умолчанию Logistic Regression использует порог $threshold = 0.5$:

$$
p \geq 0.5 \Rightarrow \hat{y}=1
$$

$$
p < 0.5 \Rightarrow \hat{y}=0
$$

Однако порог не является фиксированным свойством модели. Его можно
изменять в зависимости от задачи.

При уменьшении порога больше объектов будет отнесено к положительному
классу `Churn = 1`.

Это обычно приводит к увеличению Recall и одновременному снижению
Precision.

Проверим несколько значений порога и исследуем их влияние на качество
классификации.

In [15]:
threshold_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=1,
        class_weight=None,
        max_iter=10000
    ))
]
)

threshold_model.fit(X_train, y_train)

y_proba_threshold = threshold_model.predict_proba(X_test)[:, 1]

In [16]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

results_threshold = []

for threshold in thresholds:

    y_pred_threshold = (
        y_proba_threshold >= threshold
    ).astype(int)

    results_threshold.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_test,
            y_pred_threshold
        ),
        "Precision": precision_score(
            y_test,
            y_pred_threshold
        ),
        "Recall": recall_score(
            y_test,
            y_pred_threshold
        ),
        "F1": f1_score(
            y_test,
            y_pred_threshold
        )
    })

results_threshold = pd.DataFrame(results_threshold)

results_threshold

,Threshold,Accuracy,Precision,Recall,F1
0,0.2,0.702626,0.467153,0.855615,0.604344
1,0.3,0.749468,0.519337,0.754011,0.615049
2,0.4,0.778566,0.570776,0.668449,0.615764
3,0.5,0.806955,0.660377,0.561497,0.606936
4,0.6,0.799148,0.717703,0.401070,0.514580
5,0.7,0.766501,0.741935,0.184492,0.295503
6,0.8,0.739532,0.769231,0.026738,0.051680


### 5.1 Результаты эксперимента

Изменение порога классификации позволяет изменять соотношение между
Precision и Recall без повторного обучения модели.

При уменьшении порога увеличивается доля объектов, относимых к классу
`Churn = 1`. В результате Recall увеличивается, а Precision снижается.

На проверенном диапазоне максимальное значение F1-score получено при:

$$
threshold = 0.4
$$

При этом:

$$
Precision = 0.571
$$

$$
Recall = 0.668
$$

$$
F_1 = 0.616
$$

Для сравнения, при стандартном пороге $threshold = 0.5$:

$$
Precision = 0.660
$$

$$
Recall = 0.561
$$

$$
F_1 = 0.607
$$

Таким образом, изменение порога позволяет получить более высокий Recall
и немного увеличить F1-score без изменения обученной модели.

## 5.2 Комбинация class_weight и threshold

На предыдущем этапе использование `class_weight="balanced"` увеличило
Recall модели:

$$
Recall = 0.561 \rightarrow 0.783
$$

При этом Precision снизился:

$$
Precision = 0.660 \rightarrow 0.505
$$

Проверим, можно ли дополнительно изменить баланс между Precision и
Recall с помощью порога классификации.

Фиксируем:

$$
C=1
$$

$$
class\_weight="balanced"
$$

и проверяем несколько значений:

$$
threshold \in
\{0.2,0.3,0.4,0.5,0.6,0.7,0.8\}
$$

In [17]:
balanced_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=1,
            class_weight="balanced",
            max_iter=1000
        ))
    ]
    )

balanced_model.fit(X_train, y_train)

y_proba_balanced = balanced_model.predict_proba(X_test)[:, 1]

In [18]:
balanced_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

balanced_threshold_results = []

for threshold in balanced_thresholds:

    y_pred_balanced = (
        y_proba_balanced >= threshold
    ).astype(int)

    balanced_threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_test,
            y_pred_balanced
        ),
        "Precision": precision_score(
            y_test,
            y_pred_balanced
        ),
        "Recall": recall_score(
            y_test,
            y_pred_balanced
        ),
        "F1": f1_score(
            y_test,
            y_pred_balanced
        )
    })

balanced_threshold_results = pd.DataFrame(
    balanced_threshold_results
)

balanced_threshold_results

,Threshold,Accuracy,Precision,Recall,F1
0,0.2,0.587651,0.388350,0.962567,0.553420
1,0.3,0.656494,0.431592,0.927807,0.589134
2,0.4,0.701207,0.466187,0.866310,0.606174
3,0.5,0.738822,0.505172,0.783422,0.614256
4,0.6,0.762243,0.539877,0.705882,0.611819
5,0.7,0.789212,0.603217,0.601604,0.602410
6,0.8,0.796309,0.710145,0.393048,0.506024


### 5.3 Итог настройки Logistic Regression

В ходе экспериментов были исследованы параметры `C`, `class_weight` и
порог классификации.

Для модели без балансировки классов максимальное значение F1-score среди
проверенных порогов получено при:

$$
C=1,\quad class\_weight=None,\quad threshold=0.4
$$

При этом:

$$
Precision=0.571
$$

$$
Recall=0.668
$$

$$
F_1=0.616
$$

Использование `class_weight="balanced"` существенно увеличило Recall.
При стандартном пороге:

$$
Recall=0.783
$$

Однако максимальное значение F1 для исследованных конфигураций
осталось немного ниже:

$$
F_1=0.614
$$

Таким образом, изменение `class_weight` позволяет сместить модель в
сторону обнаружения большего количества объектов класса `Churn = 1`,
однако не приводит к увеличению F1-score относительно конфигурации
с `threshold=0.4`.

## 6. Сравнение алгоритмов

На предыдущих этапах была исследована Logistic Regression и влияние
параметров `C`, `class_weight` и `threshold`.

Теперь сравним её с другими алгоритмами классификации.

На первом этапе используются стандартные конфигурации моделей без
глубокой настройки гиперпараметров.

Будут рассмотрены:

1. Logistic Regression;
2. Decision Tree;
3. Random Forest.

Для корректного сравнения используется одна и та же обучающая и тестовая
выборка и одинаковая предварительная обработка данных там, где она
необходима.

In [19]:
logreg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=1,
            max_iter=10000
        ))
    ]
)

logreg_model.fit(X_train, y_train)

y_pred_logreg = logreg_model.predict(X_test)
y_proba_logreg = logreg_model.predict_proba(X_test)[:, 1]

In [20]:
model_results = []

model_results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_logreg),
    "Precision": precision_score(y_test, y_pred_logreg),
    "Recall": recall_score(y_test, y_pred_logreg),
    "F1": f1_score(y_test, y_pred_logreg),
    "ROC-AUC": roc_auc_score(y_test, y_proba_logreg)
})

In [21]:
tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
        random_state=42
    ))
    ]
)

tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]

model_results.append({
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, y_pred_tree),
    "Precision": precision_score(y_test, y_pred_tree),
    "Recall": recall_score(y_test, y_pred_tree),
    "F1": f1_score(y_test, y_pred_tree),
    "ROC-AUC": roc_auc_score(y_test, y_proba_tree)
})

In [22]:
forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

forest_model.fit(X_train, y_train)

y_pred_forest = forest_model.predict(X_test)
y_proba_forest = forest_model.predict_proba(X_test)[:, 1]

model_results.append({
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_forest),
    "Precision": precision_score(y_test, y_pred_forest),
    "Recall": recall_score(y_test, y_pred_forest),
    "F1": f1_score(y_test, y_pred_forest),
    "ROC-AUC": roc_auc_score(y_test, y_proba_forest)
})

In [23]:
model_results_df = pd.DataFrame(model_results)

model_results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.806955,0.660377,0.561497,0.606936,0.842171
1,Decision Tree,0.725337,0.482480,0.478610,0.480537,0.646047
2,Random Forest,0.784954,0.618729,0.494652,0.549777,0.824842


## 6.1 Результаты сравнения моделей

В стандартных конфигурациях Logistic Regression показала более высокое
качество по сравнению с Decision Tree и Random Forest.

Logistic Regression получила:

- Accuracy = 0.807;
- Precision = 0.660;
- Recall = 0.561;
- F1-score = 0.607;
- ROC-AUC = 0.842.

Decision Tree показала существенно более низкие результаты, особенно
по ROC-AUC:

$$
ROC\text{-}AUC = 0.646
$$

Random Forest показал более высокое качество по сравнению с одним
Decision Tree:

$$
ROC\text{-}AUC: 0.646 \rightarrow 0.825
$$

Однако в стандартной конфигурации Random Forest также уступает
Logistic Regression.

Полученные результаты показывают, что сравнивать модели только в
стандартных конфигурациях недостаточно. На следующем этапе будут
исследованы основные гиперпараметры деревьев и Random Forest.

## 7. Настройка Decision Tree

Одним из основных гиперпараметров дерева решений является `max_depth`.

Он ограничивает максимальную глубину дерева и тем самым контролирует
сложность модели.

Слишком маленькая глубина может привести к недообучению, поскольку
модель не сможет описать сложные зависимости в данных.

Слишком большая глубина может привести к переобучению, когда дерево
начинает подстраиваться под особенности обучающей выборки.

Проверим несколько значений:

$$
max\_depth \in \{2,3,4,5,6,8,10,None\}
$$

Значение `None` означает отсутствие ограничения максимальной глубины.

In [24]:
depth_values = [2,3,4,5,6,7,8,10, None]

tree_results = []

for depth in depth_values:
    
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", DecisionTreeClassifier(
                max_depth=depth,
                random_state=42
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred_tree = model.predict(X_test)
    y_proba_tree = model.predict_proba(X_test)[:, 1]

    tree_results.append({
        "max_depth": depth,
        "Accuracy": accuracy_score(y_test, y_pred_tree),
        "Precision": precision_score(y_test, y_pred_tree),
        "Recall": recall_score(y_test, y_pred_tree),
        "F1": f1_score(y_test, y_pred_tree),
        "ROC-AUC": roc_auc_score(y_test, y_proba_tree)
    })

tree_results = pd.DataFrame(tree_results)

tree_results

,max_depth,Accuracy,Precision,Recall,F1,ROC-AUC
0,2.0,0.787083,0.668182,0.393048,0.494949,0.777908
1,3.0,0.787083,0.668182,0.393048,0.494949,0.816766
2,4.0,0.795600,0.644295,0.513369,0.571429,0.828492
3,5.0,0.794180,0.631250,0.540107,0.582133,0.826704
4,6.0,0.800568,0.666667,0.497326,0.569678,0.821901
5,7.0,0.787793,0.618297,0.524064,0.567294,0.809608
6,8.0,0.782825,0.594444,0.572193,0.583106,0.801130
7,10.0,0.754436,0.538889,0.518717,0.528610,0.746483
8,NaN,0.725337,0.482480,0.478610,0.480537,0.646047


### 7.1 Результаты настройки max_depth

Ограничение глубины дерева существенно улучшило качество по сравнению
с моделью без ограничения глубины.

Для дерева без ограничения:

$$
ROC\text{-}AUC = 0.646
$$

При $max\_depth=4$:

$$
ROC\text{-}AUC = 0.828
$$

При дальнейшем увеличении глубины качество постепенно снижается.
Например, при $max\_depth=10$:

$$
ROC\text{-}AUC = 0.746
$$

Это показывает, что чрезмерное увеличение сложности дерева ухудшает
способность модели обобщать закономерности на тестовых данных.

Наиболее высокое значение ROC-AUC среди проверенных конфигураций
получено при $max\_depth=4$, а наиболее высокое значение F1-score —
при $max\_depth=8$.

Окончательный выбор конфигурации будет выполнен после исследования
других гиперпараметров дерева.

## 8. Настройка min_samples_leaf

Параметр `min_samples_leaf` определяет минимальное количество объектов,
которое должно находиться в каждом конечном листе дерева.

Малое значение позволяет дереву создавать очень специфические правила,
что может привести к переобучению.

Увеличение `min_samples_leaf` ограничивает сложность дерева и заставляет
модель формировать решения на основе большего количества объектов.

В данном эксперименте фиксируем:

$$
max\_depth=4
$$

и исследуем влияние:

$$
min\_samples\_leaf \in \{1,2,5,10,20,30\}
$$

In [25]:
leaf_values = [1, 2, 5, 10, 20, 30]

leaf_results = []

for leaf in leaf_values:

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=leaf,
                random_state=42
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred_leaf = model.predict(X_test)
    y_proba_leaf = model.predict_proba(X_test)[:, 1]

    leaf_results.append({
        "min_samples_leaf": leaf,
        "Accuracy": accuracy_score(y_test, y_pred_leaf),
        "Precision": precision_score(y_test, y_pred_leaf),
        "Recall": recall_score(y_test, y_pred_leaf),
        "F1": f1_score(y_test, y_pred_leaf),
        "ROC-AUC": roc_auc_score(y_test, y_proba_leaf)
    })

leaf_results = pd.DataFrame(leaf_results)

leaf_results

,min_samples_leaf,Accuracy,Precision,Recall,F1,ROC-AUC
0,1,0.7956,0.644295,0.513369,0.571429,0.828492
1,2,0.7956,0.644295,0.513369,0.571429,0.828492
2,5,0.7956,0.644295,0.513369,0.571429,0.828492
3,10,0.7956,0.644295,0.513369,0.571429,0.828475
4,20,0.7956,0.644295,0.513369,0.571429,0.828475
5,30,0.7956,0.644295,0.513369,0.571429,0.828475


### 8.1 Результаты настройки min_samples_leaf

При фиксированном значении:

$$
max\_depth=4
$$

изменение `min_samples_leaf` от 1 до 30 практически не повлияло на
результаты модели.

F1-score во всех конфигурациях составил:

$$
F_1=0.571
$$

ROC-AUC изменился только в пределах:

$$
0.828492 \rightarrow 0.828475
$$

Таким образом, при выбранной глубине дерева параметр
`min_samples_leaf` в исследованном диапазоне не оказывает существенного
влияния на качество модели.

Для дальнейшего сравнения оставляем значение:

$$
min\_samples\_leaf=1
$$


## 9. Настройка Random Forest

Random Forest представляет собой ансамбль деревьев решений.

Одним из основных параметров модели является `n_estimators` — количество
деревьев в ансамбле.

Увеличение количества деревьев позволяет получить более устойчивый
результат за счёт объединения прогнозов нескольких деревьев.

Проверим влияние количества деревьев:

$$
n\_estimators \in \{10,50,100,200,500\}
$$

Остальные параметры модели оставим без изменений.

In [26]:
forest_estimators = [10, 50, 100, 200, 500]

forest_results = []

for n_estimators in forest_estimators:

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=n_estimators,
                random_state=42
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred_forest = model.predict(X_test)
    y_proba_forest = model.predict_proba(X_test)[:, 1]

    forest_results.append({
        "n_estimators": n_estimators,
        "Accuracy": accuracy_score(y_test, y_pred_forest),
        "Precision": precision_score(y_test, y_pred_forest),
        "Recall": recall_score(y_test, y_pred_forest),
        "F1": f1_score(y_test, y_pred_forest),
        "ROC-AUC": roc_auc_score(y_test, y_proba_forest)
    })

forest_results = pd.DataFrame(forest_results)

forest_results

,n_estimators,Accuracy,Precision,Recall,F1,ROC-AUC
0,10,0.781405,0.614583,0.473262,0.534743,0.798876
1,50,0.789212,0.627907,0.505348,0.560000,0.820720
2,100,0.784954,0.618729,0.494652,0.549777,0.824842
3,200,0.789922,0.629139,0.508021,0.562130,0.825907
4,500,0.796309,0.646465,0.513369,0.572280,0.826053


## 10. Настройка max_depth в Random Forest

В Random Forest параметр `max_depth` ограничивает максимальную глубину
каждого дерева, входящего в ансамбль.

Маленькая глубина ограничивает сложность отдельных деревьев, а слишком
большая глубина позволяет деревьям формировать более сложные правила.

В предыдущем эксперименте было установлено, что увеличение количества
деревьев после 200 практически не улучшает качество. Поэтому фиксируем:

$$
n\_estimators=200
$$

и исследуем:

$$
max\_depth \in \{2,3,4,5,6,8,10,None\}
$$

где `None` означает отсутствие ограничения максимальной глубины.

In [27]:
forest_depth_values = [2, 3, 4, 5, 6, 8, 10, None]

forest_depth_results = []

for depth in forest_depth_values:

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=200,
                max_depth=depth,
                random_state=42
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred_forest_depth = model.predict(X_test)
    y_proba_forest_depth = model.predict_proba(X_test)[:, 1]

    forest_depth_results.append({
        "max_depth": depth,
        "Accuracy": accuracy_score(
            y_test,
            y_pred_forest_depth
        ),
        "Precision": precision_score(
            y_test,
            y_pred_forest_depth
        ),
        "Recall": recall_score(
            y_test,
            y_pred_forest_depth
        ),
        "F1": f1_score(
            y_test,
            y_pred_forest_depth
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            y_proba_forest_depth
        )
    })

forest_depth_results = pd.DataFrame(
    forest_depth_results
)

forest_depth_results

,max_depth,Accuracy,Precision,Recall,F1,ROC-AUC
0,2.0,0.739532,1.000000,0.018717,0.036745,0.826791
1,3.0,0.784244,0.708333,0.318182,0.439114,0.833455
2,4.0,0.787793,0.679426,0.379679,0.487136,0.837724
3,5.0,0.792761,0.679825,0.414439,0.514950,0.840735
4,6.0,0.792051,0.662651,0.441176,0.529695,0.843548
5,8.0,0.803407,0.675090,0.500000,0.574501,0.844433
6,10.0,0.805536,0.667785,0.532086,0.592262,0.843560
7,NaN,0.789922,0.629139,0.508021,0.562130,0.825907


### 10.1 Результаты настройки max_depth

Увеличение глубины деревьев Random Forest от 2 до 8 приводит к
постепенному улучшению качества модели.

ROC-AUC изменяется следующим образом:

$$
0.827 \rightarrow 0.833 \rightarrow 0.838 \rightarrow
0.841 \rightarrow 0.844
$$

При слишком малой глубине модель недообучается. Например, при
$max\_depth=2$:

$$
Recall=0.019
$$

что означает, что модель почти не обнаруживает объекты класса
`Churn = 1`.

При отсутствии ограничения глубины качество снова снижается:

$$
ROC\text{-}AUC=0.826
$$

Поэтому чрезмерное увеличение сложности отдельных деревьев также
не является оптимальным.

Лучшие результаты в исследованном диапазоне получены при глубине
8–10. Окончательная конфигурация будет определена после исследования
других параметров Random Forest.

## 11. Настройка min_samples_leaf в Random Forest

Параметр `min_samples_leaf` определяет минимальное количество
объектов обучающей выборки, которое должно находиться в каждом
конечном листе каждого дерева Random Forest.

Малое значение позволяет деревьям формировать более специфические
правила, а увеличение параметра ограничивает сложность отдельных
деревьев.

В предыдущих экспериментах было установлено, что для Random Forest
интересными являются значения глубины 8–10. Для дальнейшего
исследования фиксируем:

$$
n\_estimators=200
$$

$$
max\_depth=10
$$

и проверяем:

$$
min\_samples\_leaf \in \{1,2,5,10,20,30\}
$$

In [28]:
forest_leaf_values = [1, 2, 5, 10, 20, 30]

forest_leaf_results = []

for leaf in forest_leaf_values:

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                min_samples_leaf=leaf,
                random_state=42
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred_forest_leaf = model.predict(X_test)
    y_proba_forest_leaf = model.predict_proba(X_test)[:, 1]

    forest_leaf_results.append({
        "min_samples_leaf": leaf,
        "Accuracy": accuracy_score(
            y_test,
            y_pred_forest_leaf
        ),
        "Precision": precision_score(
            y_test,
            y_pred_forest_leaf
        ),
        "Recall": recall_score(
            y_test,
            y_pred_forest_leaf
        ),
        "F1": f1_score(
            y_test,
            y_pred_forest_leaf
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            y_proba_forest_leaf
        )
    })

forest_leaf_results = pd.DataFrame(
    forest_leaf_results
)

forest_leaf_results

,min_samples_leaf,Accuracy,Precision,Recall,F1,ROC-AUC
0,1,0.805536,0.667785,0.532086,0.592262,0.843560
1,2,0.801987,0.666667,0.508021,0.576631,0.842528
2,5,0.801987,0.664360,0.513369,0.579186,0.843406
3,10,0.801987,0.669039,0.502674,0.574046,0.844658
4,20,0.797729,0.661818,0.486631,0.560863,0.844791
5,30,0.797729,0.667925,0.473262,0.553991,0.844123


### 11.1 Результаты настройки min_samples_leaf

При фиксированных значениях:

$$
n\_estimators=200
$$

$$
max\_depth=10
$$

увеличение `min_samples_leaf` не привело к улучшению F1-score.

Максимальное значение F1-score получено при:

$$
min\_samples\_leaf=1
$$

и составляет:

$$
F_1=0.592
$$

При увеличении параметра до 30 F1-score снизился до:

$$
F_1=0.554
$$

ROC-AUC при значениях `min_samples_leaf` от 10 до 20 немного выше,
однако различия составляют около 0.001 и не являются существенными.

Таким образом, для дальнейшего сравнения оставляем:

$$
min\_samples\_leaf=1
$$

## 12. Настройка threshold для Decision Tree

После настройки структуры дерева проверим влияние порога классификации.

Фиксируем параметры:

$$
max\_depth=4
$$

$$
min\_samples\_leaf=1
$$

и исследуем:

$$
threshold \in \{0.2,0.3,0.4,0.5,0.6,0.7,0.8\}
$$

Изменение threshold не требует повторного обучения модели. Меняется
только правило преобразования вероятности в итоговый класс.

In [31]:
tree_threshold_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=1,
            random_state=42
        ))
    ]
)

tree_threshold_model.fit(X_train, y_train)

y_proba_tree_threshold = tree_threshold_model.predict_proba(X_test)[:, 1]


In [32]:
tree_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

tree_threshold_results = []

for threshold in tree_thresholds:

    y_pred = (
        y_proba_tree_threshold >= threshold
        ).astype(int)

    tree_threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

tree_threshold_results = pd.DataFrame(
    tree_threshold_results
)

tree_threshold_results

,Threshold,Accuracy,Precision,Recall,F1
0,0.2,0.665011,0.436198,0.895722,0.586690
1,0.3,0.782115,0.578824,0.657754,0.615770
2,0.4,0.782115,0.578824,0.657754,0.615770
3,0.5,0.795600,0.644295,0.513369,0.571429
4,0.6,0.779986,0.742424,0.262032,0.387352
5,0.7,0.779986,0.742424,0.262032,0.387352
6,0.8,0.760823,0.911111,0.109626,0.195704


### 12.1 Результаты настройки threshold

Изменение порога классификации существенно влияет на соотношение
Precision и Recall.

При низком пороге $threshold=0.2$ модель показывает высокий Recall:

$$
Recall=0.896
$$

однако Precision снижается:

$$
Precision=0.436
$$

Наибольшее значение F1-score среди исследованных порогов получено
при:

$$
threshold=0.3
$$

и составляет:

$$
F_1=0.616
$$

При увеличении порога Precision увеличивается, однако Recall
снижается. Например, при $threshold=0.8$:

$$
Precision=0.911
$$

$$
Recall=0.110
$$

Таким образом, изменение threshold позволяет управлять соотношением
между Precision и Recall без повторного обучения дерева.

## 13. Настройка threshold для Random Forest

После настройки гиперпараметров Random Forest исследуем влияние порога
классификации на итоговые метрики.

Фиксируем параметры модели:

$$
n\_estimators=200
$$

$$
max\_depth=10
$$

$$
min\_samples\_leaf=1
$$

Проверяем значения:

$$
threshold \in \{0.2,0.3,0.4,0.5,0.6,0.7,0.8\}
$$

Изменение threshold выполняется после обучения модели и не требует
повторного обучения.

In [33]:
forest_threshold_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=1,
            random_state=42
        ))
    ]
)

forest_threshold_model.fit(X_train, y_train)

y_proba_forest_threshold = (
    forest_threshold_model.predict_proba(X_test)[:, 1]
)

In [34]:
forest_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

forest_threshold_results = []

for threshold in forest_thresholds:

    y_pred = (
        y_proba_forest_threshold >= threshold
    ).astype(int)

    forest_threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

forest_threshold_results = pd.DataFrame(
    forest_threshold_results
)

forest_threshold_results

,Threshold,Accuracy,Precision,Recall,F1
0,0.2,0.704755,0.469477,0.863636,0.608286
1,0.3,0.757275,0.528777,0.786096,0.632258
2,0.4,0.789922,0.595122,0.652406,0.622449
3,0.5,0.805536,0.667785,0.532086,0.592262
4,0.6,0.795600,0.750000,0.344920,0.472527
5,0.7,0.775727,0.795918,0.208556,0.330508
6,0.8,0.757275,0.880952,0.098930,0.177885


### 13.1 Результаты настройки threshold

Для настроенного Random Forest было исследовано влияние порога
классификации.

При стандартном пороге:

$$
threshold=0.5
$$

получено:

$$
F_1=0.592
$$

После снижения порога до:

$$
threshold=0.3
$$

F1-score увеличился до:

$$
F_1=0.632
$$

При этом Recall увеличился:

$$
0.532 \rightarrow 0.786
$$

а Precision снизился:

$$
0.668 \rightarrow 0.529
$$

Таким образом, изменение порога позволило значительно увеличить долю
обнаруженных клиентов класса `Churn = 1`, одновременно увеличив
количество ложноположительных предсказаний.

## 14. Итоговое сравнение моделей

После проведения экспериментов сформируем финальные конфигурации
исследованных моделей.

### Logistic Regression

$$
C=1
$$

$$
class\_weight=None
$$

$$
threshold=0.4
$$

### Decision Tree

$$
max\_depth=4
$$

$$
min\_samples\_leaf=1
$$

$$
threshold=0.3
$$

### Random Forest

$$
n\_estimators=200
$$

$$
max\_depth=10
$$

$$
min\_samples\_leaf=1
$$

$$
threshold=0.3
$$

Для всех моделей используется одна и та же тестовая выборка.

In [35]:
final_models = {
    "Logistic Regression": {
        "model": LogisticRegression(
            C=1,
            class_weight=None,
            max_iter=1000
        ),
        "threshold": 0.4
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=1,
            random_state=42
        ),
        "threshold": 0.3
    },
    "Random Forest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=1,
            random_state=42
        ),
        "threshold": 0.3
    }
}

In [36]:
final_results = []

for name, config in final_models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", config["model"])
        ]
    )

    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]

    threshold = config["threshold"]

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    final_results.append({
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

final_results_df = pd.DataFrame(final_results)

final_results_df

,Model,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.4,0.778566,0.570776,0.668449,0.615764,0.842171
1,Decision Tree,0.3,0.782115,0.578824,0.657754,0.615770,0.828492
2,Random Forest,0.3,0.757275,0.528777,0.786096,0.632258,0.843560


## 14.1 Итоговое сравнение моделей

После настройки гиперпараметров и порога классификации было проведено
итоговое сравнение трёх моделей.

Для Logistic Regression использовались:

$$
C=1
$$

$$
class\_weight=None
$$

$$
threshold=0.4
$$

Для Decision Tree:

$$
max\_depth=4
$$

$$
min\_samples\_leaf=1
$$

$$
threshold=0.3
$$

Для Random Forest:

$$
n\_estimators=200
$$

$$
max\_depth=10
$$

$$
min\_samples\_leaf=1
$$

$$
threshold=0.3
$$

Итоговые результаты показывают различия в соотношении Precision и
Recall между моделями.

Random Forest при выбранном пороге имеет Recall:

$$
Recall=0.786
$$

и F1-score:

$$
F_1=0.632
$$

Logistic Regression показывает:

$$
F_1=0.616
$$

и ROC-AUC:

$$
ROC\text{-}AUC=0.842
$$

Decision Tree показывает:

$$
F_1=0.616
$$

и ROC-AUC:

$$
ROC\text{-}AUC=0.828
$$

Таким образом, проведённые эксперименты показали, что изменение
гиперпараметров и порога классификации существенно влияет на
соотношение Precision и Recall. При этом значения ROC-AUC у Logistic
Regression и Random Forest остаются близкими.

## 15. Автоматический подбор гиперпараметров Logistic Regression

После ручного исследования параметров Logistic Regression применим
`GridSearchCV` для автоматического поиска оптимальной комбинации
гиперпараметров.

Будут исследованы:

$$
C \in \{0.01,0.1,1,10,100\}
$$

и:

$$
class\_weight \in \{None,balanced\}
$$

Всего будет проверено:

$$
5 \times 2 = 10
$$

комбинаций.

Для оценки каждой комбинации используется 5-fold cross-validation.

В качестве целевой метрики оптимизации используется F1-score.

Тестовая выборка не используется при подборе гиперпараметров.

In [ ]:
from sklearn.model_selection import GridSearchCV

logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=10000
        ))
    ]
)

param_grid_lr = {
    "model__C":  [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

grid_lr = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=param_grid_lr,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)